# JARVIS AI — Hardware and Google Drive Audit

Run this notebook before selecting or training a model. It checks the actual Colab runtime, mounts Google Drive with your approval, creates a permanent JARVIS_AI directory, clones the public JARVIS repository, and saves a hardware report. **It does not scrape the internet or train a model.**

In [ ]:
from pathlib import Path
import json, os, platform, shutil, subprocess, sys
from datetime import datetime, timezone

try:
    import torch
except ImportError as exc:
    raise RuntimeError('PyTorch is not available in this runtime. Select a standard Colab Python runtime and retry.') from exc

try:
    import psutil
    ram_total_gb = round(psutil.virtual_memory().total / (1024**3), 2)
    ram_available_gb = round(psutil.virtual_memory().available / (1024**3), 2)
except ImportError:
    ram_total_gb = None
    ram_available_gb = None

gpu_available = torch.cuda.is_available()
gpu_name = torch.cuda.get_device_name(0) if gpu_available else None
gpu_vram_gb = round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2) if gpu_available else 0
disk = shutil.disk_usage('/content')

report = {
    'created_at': datetime.now(timezone.utc).isoformat(),
    'python': sys.version,
    'platform': platform.platform(),
    'pytorch': torch.__version__,
    'cuda_available': gpu_available,
    'gpu_name': gpu_name,
    'gpu_vram_gb': gpu_vram_gb,
    'ram_total_gb': ram_total_gb,
    'ram_available_gb': ram_available_gb,
    'disk_total_gb': round(disk.total / (1024**3), 2),
    'disk_free_gb': round(disk.free / (1024**3), 2)
}
print(json.dumps(report, indent=2))

## Mount Google Drive

Run this cell and approve Google's Drive access prompt. This is where permanent datasets, checkpoints, adapters and reports will be stored.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import json

DRIVE_ROOT = Path('/content/drive/MyDrive/JARVIS_AI')
folders = [
    'project_sources', 'source_manifests', 'approved_corpus', 'derived_notes',
    'datasets/train', 'datasets/validation', 'datasets/test',
    'runs', 'checkpoints', 'adapters', 'evaluation_reports', 'backups'
]
for folder in folders:
    (DRIVE_ROOT / folder).mkdir(parents=True, exist_ok=True)

report_path = DRIVE_ROOT / 'hardware_report.json'
report_path.write_text(json.dumps(report, indent=2), encoding='utf-8')
print('Created:', DRIVE_ROOT)
print('Hardware report:', report_path)
print('Folders:')
for folder in folders:
    print(' -', folder)

## Clone the JARVIS repository

The repository is public, so no GitHub token is needed. Do not paste a personal access token into this notebook.

In [ ]:
REPO_URL = 'https://github.com/18110priyanshuraj-creator/Jarvis.git'
REPO_DIR = Path('/content/Jarvis')
if not (REPO_DIR / '.git').exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)
else:
    print('Repository already exists; leaving it unchanged.')

print('Repository:', REPO_DIR)
print('Training workspace:', REPO_DIR / 'ai-training')
print('TinyGPT experiment:', REPO_DIR / 'experiments' / 'tiny-gpt')

## Next step

Keep the hardware report in Drive. The next notebook should select a model only after reviewing its license, tokenizer/chat template, supported training stack and the measured VRAM. Colab resource availability can change; do not assume a GPU will always be assigned.